In [1]:
import pandas as pd
path_data_dir = "../../data/preprocessed/" 
train_data_file_pp = path_data_dir + "hoteles_train_preprocessed.csv" 
test_data_file_pp = path_data_dir + "hoteles_test_preprocessed.csv"
#train data reanding
train_data_pp = pd.read_csv(train_data_file_pp, sep=",")
print('')
print('train data info:')
train_data_pp.info()
test_data_pp = pd.read_csv(test_data_file_pp, sep=",")
print('')
print('test data info:')
test_data_pp.info()


train data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52981 entries, 0 to 52980
Data columns (total 34 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           52981 non-null  int64  
 1   lead_time                       52981 non-null  int64  
 2   stays_in_weekend_nights         52981 non-null  int64  
 3   stays_in_week_nights            52981 non-null  int64  
 4   adults                          52981 non-null  int64  
 5   children                        52981 non-null  int64  
 6   country                         52681 non-null  object 
 7   market_segment                  52981 non-null  object 
 8   distribution_channel            52981 non-null  object 
 9   is_repeated_guest               52981 non-null  int64  
 10  previous_cancellations          52981 non-null  int64  
 11  previous_bookings_not_canceled  52981 non-null  int64  
 12  reserved_room_

Optimization of Hyoperparameters for XGBClassifier

Create a Pipeline for preprocessing and modeling

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import os
train_data = train_data_pp
test_data = test_data_pp
# Define features (exclude 'children')
X = train_data.drop(columns=['children', 'has_children'])
y = train_data['has_children']
# Split categorical/numerical
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(exclude=['object']).columns.tolist()

# Imputers (no data evalable set median value for numerical, most_frequent for categorical)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
 
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Model
clf_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# Train/test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit
clf_xgb.fit(X_train, y_train)
# add the new accuracy value to the csv, including the number of validation samples
val_acc_file = '../../data/outputs/val_acc.csv'
val_acc = clf_xgb.score(X_val, y_val)

if os.path.exists(val_acc_file):
    df_acc = pd.read_csv(val_acc_file)
    # Get the number of validation samples
    val_n = df_acc['val_n'].iloc[-1]
    # Append new row with correct val_n
    method = 'XGBCalssifier'
    df_acc = pd.concat([df_acc, pd.DataFrame([{'val_acc': val_acc, 'val_n': val_n+1, 'method':  method}])], ignore_index=True)
    df_acc = pd.concat([df_acc, pd.DataFrame([{'val_acc': val_acc, 'val_n': val_n+1}])], ignore_index=True)
else:
    df_acc = pd.DataFrame({'val_acc': [val_acc], 'val_n': 1})

df_acc.to_csv(val_acc_file, index=False)

clf_xgb.fit(X_train, y_train)
y_pred = clf_xgb.predict(X_val)
print("Accuracy:", clf_xgb.score(X_val, y_val))


/home/trudolf/anaconda3/envs/ml2025/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/trudolf/anaconda3/envs/ml2025/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [07:44:24] WARNING: /croot/xgboost-split_1749630910898/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/trudolf/anaconda3/envs/ml2025/lib/python3.11/site-packages/xgboost/training.py:183: UserWarni

Accuracy: 0.9480041521185241


In [3]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

search_spaces = {
    'classifier__n_estimators': Integer(100, 500),
    'classifier__max_depth': Integer(3, 10),
    'classifier__learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'classifier__subsample': Real(0.5, 1.0)
}

opt_xgb = BayesSearchCV(
    clf_xgb,
    search_spaces,
    n_iter=30,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

opt_xgb.fit(X_train, y_train)

print("Best parameters:", opt_xgb.best_params_)
print("Best validation accuracy:", opt_xgb.best_score_)


/home/trudolf/anaconda3/envs/ml2025/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/trudolf/anaconda3/envs/ml2025/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux201

Best parameters: OrderedDict([('classifier__learning_rate', 0.046333474421394044), ('classifier__max_depth', 10), ('classifier__n_estimators', 386), ('classifier__subsample', 0.5)])
Best validation accuracy: 0.9488486221215554


In [4]:
#  prep test data for pipeline}
df_test = test_data.copy()

# Predict on validation set
y_pred = clf_xgb.predict(df_test)


In [5]:
# Predict on new data
X_test = df_test.copy()
y_pred = opt_xgb.best_estimator_.predict(X_test)



In [6]:
val_acc = opt_xgb.best_score_
if os.path.exists(val_acc_file):
    df_acc = pd.read_csv(val_acc_file)
    # Get the number of validation samples
    val_n = df_acc['val_n'].iloc[-1]
    # Append new row with correct val_n
    method = 'XGBClassifier, BO'
    df_acc = pd.concat([df_acc, pd.DataFrame([{'val_acc': val_acc, 'val_n': val_n+1, 'method':  method}])], ignore_index=True)

else:
    df_acc = pd.DataFrame({'val_acc': [val_acc], 'val_n': 1})

df_acc.to_csv(val_acc_file, index=False)


In [7]:
# Predict probabilities for validation set (y=1 means has children)
#y_proba = clf_xgb.predict_proba(df_test)[:, 1]

y_proba = opt_xgb.best_estimator_.predict_proba(df_test)
X_test = df_test.copy()
X_test['id'] = range(1, len(df_test) + 1)

In [8]:
# Create a  DataFrame for submissio 
df_submission = pd.DataFrame({
    'id': X_test['id'],
    'prob': y_proba[:, 1]
})
df_submission.head()

,id,prob
0,1,0.004678
1,2,0.075196
2,3,0.721993
3,4,0.050069
4,5,0.017005


In [9]:
from datetime import datetime
date = datetime.now().strftime('%y%m%d')  # format yy_mm_dd
new_sumbission_file = f'../../data/outputs/submission_rdf_{date}_xgb-bo.csv'

df_submission.to_csv(new_sumbission_file, index=False)
print(f'sumbission saved to: {new_sumbission_file}')

sumbission saved to: ../../data/outputs/submission_rdf_251005_xgb-bo.csv
